In [ ]:
#RF01 – Criar ou Carregar o Dataset de Vendas
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

def gerar_dataset_vendas(n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
                "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                  "Tablet": "Celulares", "Monitor": "Computadores",
                  "Teclado": "Perifericos", "Mouse": "Perifericos",
                  "Headset": "Perifericos"}
    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
              "Monitor": 1200, "Teclado": 250, "Mouse": 120,
              "Headset": 350}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]

    data_inicio = datetime(2025, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        categoria = categorias[produto]
        quantidade = random.randint(1, 10)
        preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))
        data_txt = data.strftime("%Y-%m-%d")
        cliente = f"Cliente_{random.randint(1, 50):03d}"

        # --- sujeira proposital para a etapa de limpeza ---
        if random.random() < 0.03:
        	data_txt = "DATA INVALIDA"       	# data invalida

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_txt,
            "cliente": cliente,
            "produto": produto,
            "categoria": categoria,
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco,
        })

    return pd.DataFrame(dados)


# Gerar e salvar o CSV bruto
df_bruto = gerar_dataset_vendas()
df_bruto.to_csv("vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")
print(df_bruto.head())

Dataset gerado com 200 registros.
   id_venda     data_venda      cliente     produto     categoria    regiao  \
0         1     2025-05-21  Cliente_016       Mouse   Perifericos   Sudeste   
1         2  DATA INVALIDA  Cliente_003       Mouse   Perifericos       Sul   
2         3     2025-10-15  Cliente_013  Smartphone     Celulares     Norte   
3         4     2025-05-23  Cliente_001     Monitor  Computadores       Sul   
4         5     2025-03-21  Cliente_014       Mouse   Perifericos  Nordeste   

   quantidade  preco_unitario  
0           2          102.90  
1           9          105.13  
2           9         2267.33  
3           4         1181.72  
4           7          114.25  


In [ ]:
#RF02 – Inspecionar e Descrever os Dados
def inspecionar_dados(df):
    """Exibe as informacoes estruturais do DataFrame."""
    print("\nINSPECAO INICIAL DO DATASET:")
    print(f"\nShape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:\n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df.head()}")
    return df
  # Carrega o CSV
df_bruto = pd.read_csv("vendas.csv")

# Chama a função passando a tabela
inspecionar_dados(df_bruto)


INSPECAO INICIAL DO DATASET:

Shape: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade          int64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        0
preco_unitario    0
dtype: int64

Primeiros registros:
   id_venda     data_venda      cliente     produto     categoria    regiao  \
0         1     2025-05-21  Cliente_016       Mouse   Perifericos   Sudeste   
1         2  DATA INVALIDA  Cliente_003       Mouse   Perifericos       Sul   
2         3     2025-10-15  Cliente_013  Smartphone     Celulares     Norte   
3         4     2025-05-23  Cliente_001     Monitor  Computado

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,Cliente_016,Mouse,Perifericos,Sudeste,2,102.90
1,2,DATA INVALIDA,Cliente_003,Mouse,Perifericos,Sul,9,105.13
2,3,2025-10-15,Cliente_013,Smartphone,Celulares,Norte,9,2267.33
3,4,2025-05-23,Cliente_001,Monitor,Computadores,Sul,4,1181.72
4,5,2025-03-21,Cliente_014,Mouse,Perifericos,Nordeste,7,114.25
...,...,...,...,...,...,...,...,...
195,196,2025-04-21,Cliente_028,Headset,Perifericos,Sul,9,373.19
196,197,2025-07-24,Cliente_027,Monitor,Computadores,Nordeste,6,1317.87
197,198,2025-07-11,Cliente_010,Monitor,Computadores,Centro-Oeste,6,1259.44
198,199,2025-02-17,Cliente_028,Notebook,Computadores,Nordeste,2,3848.36


In [ ]:
#RF03 – Tratar os Dados (datetime)

def tratar_dados(df):
  """Trata o DataFrame de vendas:

  1. Remove espaços extras em colunas de texto.
  2. Converte 'data_venda' para datetime e descarta datas inválidas.
  3. Converte 'quantidade' para int e 'preco_unitario' para float.
  4. Exibe e retorna o relatório de limpeza.
  """


  df_limpo = df.copy()
  qtd_inicial = len(df_limpo)

  # 1. Remover espaços extras em colunas de texto (tipo 'object')
  colunas_texto = df_limpo.select_dtypes(include=["object"]).columns
  for col in colunas_texto:
    df_limpo[col] = df_limpo[col].astype(str).str.strip()

  # 2. Converter data_venda para datetime (invalidos passam a ser NaT)
  df_limpo["data_venda"] = pd.to_datetime(
      df_limpo["data_venda"], errors="coerce"
  )

  # Contar quantos registros eram datas inválidas
  qtd_datas_invalidas = df_limpo["data_venda"].isnull().sum()

  # Descartar linhas onde a data virou NaT
  df_limpo = df_limpo.dropna(subset=["data_venda"]).reset_index(drop=True)

  # 3. Garantir os tipos numéricos corretos
  df_limpo["quantidade"] = df_limpo["quantidade"].astype(int)
  df_limpo["preco_unitario"] = df_limpo["preco_unitario"].astype(float)

  qtd_final = len(df_limpo)

  # 4. Montar e exibir o relatório de limpeza
  relatorio = {
      "registros_iniciais": qtd_inicial,
      "datas_invalidas_removidas": qtd_datas_invalidas,
      "registros_finais": qtd_final,
  }

  print("RELATÓRIO DE LIMPEZA E TRATAMENTO:")
  print(f"\nRegistros recebidos: {relatorio['registros_iniciais']}")
  print(
      "Datas inválidas removidas:"
      f" {relatorio['datas_invalidas_removidas']}"
  )
  print(f"Registros válidos restantes: {relatorio['registros_finais']}")

  return df_limpo, relatorio

# Carregar os dados brutos
df_bruto = pd.read_csv("vendas.csv")

# Executar a limpeza
df_limpo, relatorio_limpeza = tratar_dados(df_bruto)

# Conferir se o tipo da coluna mudou para datetime64
print("\nNovo tipo da coluna data_venda:", df_limpo["data_venda"].dtype)

# Ver o df limpo
df_limpo

RELATÓRIO DE LIMPEZA E TRATAMENTO:

Registros recebidos: 200
Datas inválidas removidas: 6
Registros válidos restantes: 194

Novo tipo da coluna data_venda: datetime64[ns]


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,Cliente_016,Mouse,Perifericos,Sudeste,2,102.90
1,3,2025-10-15,Cliente_013,Smartphone,Celulares,Norte,9,2267.33
2,4,2025-05-23,Cliente_001,Monitor,Computadores,Sul,4,1181.72
3,5,2025-03-21,Cliente_014,Mouse,Perifericos,Nordeste,7,114.25
4,6,2025-07-03,Cliente_023,Notebook,Computadores,Sudeste,2,3373.92
...,...,...,...,...,...,...,...,...
189,196,2025-04-21,Cliente_028,Headset,Perifericos,Sul,9,373.19
190,197,2025-07-24,Cliente_027,Monitor,Computadores,Nordeste,6,1317.87
191,198,2025-07-11,Cliente_010,Monitor,Computadores,Centro-Oeste,6,1259.44
192,199,2025-02-17,Cliente_028,Notebook,Computadores,Nordeste,2,3848.36


In [ ]:
colunas_texto = df_bruto.select_dtypes(include=['object']).columns

for col in colunas_texto:
  # Compara a coluna original com a versão com strip
  com_espaco = (df_bruto[col] != df_bruto[col].astype(str).str.strip()).sum()
  print(f"Coluna '{col}': {com_espaco} valores tinham espaços extras.")

Coluna 'data_venda': 0 valores tinham espaços extras.
Coluna 'cliente': 0 valores tinham espaços extras.
Coluna 'produto': 0 valores tinham espaços extras.
Coluna 'categoria': 0 valores tinham espaços extras.
Coluna 'regiao': 0 valores tinham espaços extras.


In [ ]:
#RF04 – Criar Colunas Derivadas com Transformações Condicionais

def criar_colunas_derivadas(df):
  """Cria colunas derivadas para análise de vendas.

  - receita_total: quantidade * preco_unitario
  - mes, mes_nome, trimestre, ano: componentes temporais da data_venda
  - faixa_receita_item: classificação condicional da receita por linha
  """
  df_trans = df.copy()

  # 1. Receita total por linha (operação vetorizada)
  df_trans["receita_total"] = (
      df_trans["quantidade"] * df_trans["preco_unitario"]
  ).round(2)

  # 2. Componentes temporais da data (.dt)
  df_trans["mes"] = df_trans["data_venda"].dt.month
  df_trans["ano"] = df_trans["data_venda"].dt.year
  df_trans["trimestre"] = df_trans["data_venda"].dt.quarter.apply(
      lambda q: f"Q{q}"
  )

  # Mapeamento do número do mês para o nome em português
  meses_pt = {
      1: "Janeiro",
      2: "Fevereiro",
      3: "Março",
      4: "Abril",
      5: "Maio",
      6: "Junho",
      7: "Julho",
      8: "Agosto",
      9: "Setembro",
      10: "Outubro",
      11: "Novembro",
      12: "Dezembro",
  }
  df_trans["mes_nome"] = df_trans["mes"].map(meses_pt)

  # 3. Transformação condicional para a Faixa de Receita do Item
  df_trans["faixa_receita_item"] = "Não Classificado"

  condicoes = [
      df_trans["receita_total"] < 500,
      (df_trans["receita_total"] >= 500) & (df_trans["receita_total"] < 5000),
      df_trans["receita_total"] >= 5000,
  ]
  faixas = ["Baixo Valor", "Medio Valor", "Alto Valor"]

  for condicao, faixa in zip(condicoes, faixas):
    df_trans.loc[condicao, "faixa_receita_item"] = faixa

  print("\n")
  print("COLUNAS DERIVADAS CRIADAS COM SUCESSO:")
  print("\n")
  print(
      "Novas colunas adicionadas:"
      " ['receita_total', 'mes', 'mes_nome', 'trimestre', 'ano',"
      " 'faixa_receita_item']"
  )

  return df_trans


# EXECUÇÃO E VERIFICAÇÃO NO COLAB

# 1. Aplica as transformações sobre a tabela tratada (df_limpo)
df_transformado = criar_colunas_derivadas(df_limpo)

# 2. Exibe as 5 primeiras linhas selecionando todas as colunas temporais e calculadas
df_transformado[[
    "data_venda",
    "ano",
    "mes",
    "mes_nome",
    "trimestre",
    "quantidade",
    "preco_unitario",
    "receita_total",
    "faixa_receita_item",
]].head()



COLUNAS DERIVADAS CRIADAS COM SUCESSO:


Novas colunas adicionadas: ['receita_total', 'mes', 'mes_nome', 'trimestre', 'ano', 'faixa_receita_item']


,data_venda,ano,mes,mes_nome,trimestre,quantidade,preco_unitario,receita_total,faixa_receita_item
0,2025-05-21,2025,5,Maio,Q2,2,102.90,205.80,Baixo Valor
1,2025-10-15,2025,10,Outubro,Q4,9,2267.33,20405.97,Alto Valor
2,2025-05-23,2025,5,Maio,Q2,4,1181.72,4726.88,Medio Valor
3,2025-03-21,2025,3,Março,Q1,7,114.25,799.75,Medio Valor
4,2025-07-03,2025,7,Julho,Q3,2,3373.92,6747.84,Alto Valor


In [ ]:
#RF05 – Calcular Métricas Agregadas com groupby
import pandas as pd


def calcular_metricas(df):
  """Calcula as métricas agregadas do dataset (RF05).

  Retorna um dicionário no formato {nome_da_metrica: DataFrame}:
  - por_mes: receita total, quantidade e número de vendas por mês
  - top_produtos: top 5 produtos por receita total
  - por_categoria: receita total por categoria
  - por_regiao: receita total e ticket médio por região
  """
  metricas = {}

  # 1. Agregação Por Mês (Receita Total, Quantidade e Nº de Vendas)
  por_mes = (
      df.groupby(["mes"])
      .agg(
          receita_total=("receita_total", "sum"),
          quantidade=("quantidade", "sum"),
          n_vendas=("id_venda", "count"),
      )
      .reset_index()
  )
  metricas["por_mes"] = por_mes

  # 2. Top 5 Produtos por Receita Total
  top_produtos = (
      df.groupby("produto")
      .agg(receita_total=("receita_total", "sum"))
      .sort_values(by="receita_total", ascending=False)
      .head(5)
      .reset_index()
  )
  metricas["top_produtos"] = top_produtos

  # 3. Agregação Por Categoria
  por_categoria = (
      df.groupby("categoria")
      .agg(receita_total=("receita_total", "sum"))
      .sort_values(by="receita_total", ascending=False)
      .reset_index()
  )
  metricas["por_categoria"] = por_categoria

  # 4. Agregação Por Região (Receita Total e Ticket Médio)
  por_regiao = (
      df.groupby("regiao")
      .agg(
          receita_total=("receita_total", "sum"),
          ticket_medio=("receita_total", "mean"),
      )
      .sort_values(by="receita_total", ascending=False)
      .reset_index()
  )
  # Arredondar valores decimais monetários
  por_regiao["ticket_medio"] = por_regiao["ticket_medio"].round(2)
  metricas["por_regiao"] = por_regiao

  # Exibição formatada no console
  print("\n")
  print("MÉTRICAS AGREGADAS:")
  print("\n")

  for chave, df_metrica in metricas.items():
    print(f"\n {chave.upper().replace('_', ' ')}:\n")
    print(df_metrica.to_string(index=False))

  return metricas

  # Executa as agregações do RF05
dicionario_metricas = calcular_metricas(df_transformado)



MÉTRICAS AGREGADAS:



 POR MES:

 mes  receita_total  quantidade  n_vendas
   1      121707.77          86        15
   2      186065.22          84        18
   3      129138.46          87        19
   4       68247.67          85        14
   5      103600.61          75        12
   6       72067.26          73        16
   7      129427.58          97        19
   8      156072.31         107        19
   9       57509.42          77        14
  10      181878.41         137        25
  11       89645.62          86        15
  12       84849.98          41         8

 TOP PRODUTOS:

   produto  receita_total
  Notebook      493242.63
Smartphone      343721.95
    Tablet      236519.71
   Monitor      205212.61
   Headset       48879.79

 POR CATEGORIA:

   categoria  receita_total
Computadores      698455.24
   Celulares      580241.66
 Perifericos      101513.41

 POR REGIAO:

      regiao  receita_total  ticket_medio
Centro-Oeste      342981.92       7297.49
       Norte    

In [ ]:
#RF06 – Segmentar Clientes por Nível de Gasto

def segmentar_clientes(df):
  """Agrupa por cliente, calcula o total gasto e classifica em Bronze, Prata e Ouro.

  Utiliza função lambda com .apply() conforme requisito.
  """
  # 1. Agrupar por cliente e somar a receita total
  df_clientes = (
      df.groupby("cliente")
      .agg(total_gasto=("receita_total", "sum"))
      .sort_values(by="total_gasto", ascending=False)
      .reset_index()
  )

  # Arredondar valores para 2 casas decimais
  df_clientes["total_gasto"] = df_clientes["total_gasto"].round(2)

  # 2. Classificação usando Função Lambda com .apply()
  df_clientes["segmento"] = df_clientes["total_gasto"].apply(
      lambda gasto: (
          "Ouro" if gasto > 15000 else ("Prata" if gasto >= 5000 else "Bronze")
      )
  )

  # 3. Exibição dos resultados no console
  print("\n")
  print("SEGMENTAÇÃO DE CLIENTES:")

  print("\nTop 10 Maiores Clientes:\n")
  print(df_clientes.head(10).to_string(index=False))

  print("\nDistribuição de Clientes por Segmento:\n")
  print(df_clientes["segmento"].value_counts().to_string())

  return df_clientes

  # Executa a segmentação de clientes
df_clientes_segmentados = segmentar_clientes(df_transformado)



SEGMENTAÇÃO DE CLIENTES:

Top 10 Maiores Clientes:

    cliente  total_gasto segmento
Cliente_049     86381.62     Ouro
Cliente_006     78524.09     Ouro
Cliente_009     72517.27     Ouro
Cliente_016     66606.02     Ouro
Cliente_008     58695.74     Ouro
Cliente_020     54182.88     Ouro
Cliente_044     54128.61     Ouro
Cliente_028     51710.71     Ouro
Cliente_038     44741.97     Ouro
Cliente_035     43323.63     Ouro

Distribuição de Clientes por Segmento:

segmento
Ouro      35
Prata      9
Bronze     5


In [ ]:
#RF07 – Criar Visualizações com Matplotlib

import matplotlib.pyplot as plt
import seaborn as sns


def gerar_graficos(df, metricas):
  """Gera e salva os 4 gráficos exigidos no RF07."""
  sns.set_theme(style="whitegrid")
  plt.rcParams["font.sans-serif"] = "DejaVu Sans"

  # 1. Gráfico de Linha: Receita Mensal

  fig, ax = plt.subplots(figsize=(10, 5))
  df_mes = metricas["por_mes"]
  ax.plot(
      df_mes["mes"],
      df_mes["receita_total"],
      marker="o",
      color="#1f77b4",
      linewidth=2.5,
      markersize=8,
  )
  ax.set_title(
      "Evolução da Receita Mensal (2025)", fontsize=14, fontweight="bold", pad=15
  )
  ax.set_xlabel("Mês (Número)", fontsize=11)
  ax.set_ylabel("Receita Total (R$)", fontsize=11)
  ax.set_xticks(df_mes["mes"])
  plt.tight_layout()
  plt.savefig("receita_por_mes.png", dpi=300, bbox_inches="tight")
  plt.close()

  # 2. Gráfico de Barras: Top 5 Produtos

  fig, ax = plt.subplots(figsize=(9, 5))
  df_prod = metricas["top_produtos"]
  sns.barplot(
      data=df_prod,
      x="receita_total",
      y="produto",
      palette="Blues_r",
      ax=ax,
      hue="produto",
      legend=False,
  )
  ax.set_title(
      "Top 5 Produtos por Receita Total", fontsize=14, fontweight="bold", pad=15
  )
  ax.set_xlabel("Receita Total (R$)", fontsize=11)
  ax.set_ylabel("Produto", fontsize=11)
  plt.tight_layout()
  plt.savefig("top_produtos.png", dpi=300, bbox_inches="tight")
  plt.close()

  # 3. Gráfico de Dispersão: Quantidade vs Receita

  fig, ax = plt.subplots(figsize=(9, 5))
  sns.scatterplot(
      data=df,
      x="quantidade",
      y="receita_total",
      hue="categoria",
      style="categoria",
      s=70,
      ax=ax,
  )
  ax.set_title(
      "Relação entre Quantidade e Receita Total por Venda",
      fontsize=14,
      fontweight="bold",
      pad=15,
  )
  ax.set_xlabel("Quantidade do Item", fontsize=11)
  ax.set_ylabel("Receita da Venda (R$)", fontsize=11)
  plt.tight_layout()
  plt.savefig("quantidade_vs_receita.png", dpi=300, bbox_inches="tight")
  plt.close()

  # 4. Painel Resumo (Subplots 2x2)

  fig, axes = plt.subplots(2, 2, figsize=(14, 10))
  fig.suptitle(
      "PAINEL RESUMO - SALESINSIGHT PY", fontsize=16, fontweight="bold", y=0.98
  )

  # Subplot (0, 0): Linha - Receita Mensal
  axes[0, 0].plot(
      df_mes["mes"],
      df_mes["receita_total"],
      marker="s",
      color="#2ca02c",
      linewidth=2,
  )
  axes[0, 0].set_title("Receita por Mês")
  axes[0, 0].set_xlabel("Mês")
  axes[0, 0].set_ylabel("Receita (R$)")
  axes[0, 0].set_xticks(df_mes["mes"])

  # Subplot (0, 1): Barras - Top 5 Produtos
  sns.barplot(
      data=df_prod,
      x="receita_total",
      y="produto",
      ax=axes[0, 1],
      palette="Greens_r",
      hue="produto",
      legend=False,
  )
  axes[0, 1].set_title("Top 5 Produtos")
  axes[0, 1].set_xlabel("Receita (R$)")

  # Subplot (1, 0): Dispersão - Quantidade vs Receita
  sns.scatterplot(
      data=df,
      x="quantidade",
      y="receita_total",
      hue="categoria",
      s=50,
      ax=axes[1, 0],
  )
  axes[1, 0].set_title("Quantidade vs Receita")
  axes[1, 0].set_xlabel("Quantidade")
  axes[1, 0].set_ylabel("Receita (R$)")

  # Subplot (1, 1): Barras - Receita por Região
  df_reg = metricas["por_regiao"]
  sns.barplot(
      data=df_reg,
      x="regiao",
      y="receita_total",
      ax=axes[1, 1],
      palette="Oranges_r",
      hue="regiao",
      legend=False,
  )
  axes[1, 1].set_title("Receita por Região")
  axes[1, 1].set_xlabel("Região")
  axes[1, 1].set_ylabel("Receita (R$)")
  axes[1, 1].tick_params(axis="x", rotation=30)

  plt.tight_layout(rect=[0, 0.03, 1, 0.95])
  plt.savefig("painel_resumo.png", dpi=300, bbox_inches="tight")
  plt.close()

  print("\n")
  print("GRÁFICOS GERADOS E SALVOS COM SUCESSO:")
  print("\n")
  print("Arquivos salvos:")
  print("  1. receita_por_mes.png")
  print("  2. top_produtos.png")
  print("  3. quantidade_vs_receita.png")
  print("  4. painel_resumo.png")


# Executa para gerar e atualizar todos os arquivos PNG
gerar_graficos(df_transformado, dicionario_metricas)



GRÁFICOS GERADOS E SALVOS COM SUCESSO:


Arquivos salvos:
  1. receita_por_mes.png
  2. top_produtos.png
  3. quantidade_vs_receita.png
  4. painel_resumo.png


In [ ]:
# RF08 – Organizar o Código em Funções Reutilizáveis

import pandas as pd

def processar_coluna(df, coluna, funcao_transformacao, nome_saida=None):
  """Aplica uma função de transformação a uma coluna do DataFrame.

  Demonstra o conceito de Função de Ordem Superior,
  recebendo uma função como argumento.
  """
  df_mod = df.copy()
  nome_saida = nome_saida or f"{coluna}_transformado"

  # Aplica a função recebida como parâmetro na coluna indicada
  df_mod[nome_saida] = df_mod[coluna].apply(funcao_transformacao)
  return df_mod


# EXECUÇÃO E TESTE DA FUNÇÃO DE ORDEM SUPERIOR:

# Exemplo 1: Converter receita_total para milhares (ex: 5500.00 -> 5.5)
df_transformado = processar_coluna(
    df_transformado,
    coluna="receita_total",
    funcao_transformacao=lambda x: round(x / 1000, 2),
    nome_saida="receita_em_milhares",
)

# Exemplo 2: Classificar o volume da venda com base na quantidade
df_transformado = processar_coluna(
    df_transformado,
    coluna="quantidade",
    funcao_transformacao=lambda q: "Alto Volume" if q > 5 else "Baixo Volume",
    nome_saida="perfil_volume",
)

# Exibe o resultado das colunas criadas para conferência
print("\n")
print("FUNÇÃO DE ORDEM SUPERIOR APLICADA:")
print("\n")
print(
    df_transformado[[
        "quantidade",
        "perfil_volume",
        "receita_total",
        "receita_em_milhares",
    ]].head(10)
)



FUNÇÃO DE ORDEM SUPERIOR APLICADA:


   quantidade perfil_volume  receita_total  receita_em_milhares
0           2  Baixo Volume         205.80                 0.21
1           9   Alto Volume       20405.97                20.41
2           4  Baixo Volume        4726.88                 4.73
3           7   Alto Volume         799.75                 0.80
4           2  Baixo Volume        6747.84                 6.75
5           8   Alto Volume         970.40                 0.97
6           6   Alto Volume        2148.72                 2.15
7           5  Baixo Volume        2004.75                 2.00
8           6   Alto Volume         647.16                 0.65
9           3  Baixo Volume         757.68                 0.76


In [ ]:
#RF09 – Exportar Resultados em CSV e JSON

import json
import pandas as pd


def exportar_resultados(metricas, df_clientes, df_transformado):
  """Exporta os relatórios em CSV e as estatísticas gerais em JSON (RF09).

  Responde às principais perguntas de negócio e comprova a releitura.
  """
  # 1. Exportar Tabelas para CSV
  metricas["por_mes"].to_csv(
      "metricas_por_mes.csv", index=False, encoding="utf-8-sig"
  )
  df_clientes.to_csv(
      "segmentacao_clientes.csv", index=False, encoding="utf-8-sig"
  )

  # 2. Montar Dicionário com Estatísticas Gerais e Insights de Negócio
  df_mes = metricas["por_mes"]
  mes_top = df_mes.sort_values("receita_total", ascending=False).iloc[0]
  prod_top = metricas["top_produtos"].iloc[0]
  cat_top = metricas["por_categoria"].sort_values(
      "receita_total", ascending=False
  ).iloc[0]
  reg_top = metricas["por_regiao"].sort_values(
      "receita_total", ascending=False
  ).iloc[0]
  reg_ticket_top = metricas["por_regiao"].sort_values(
      "ticket_medio", ascending=False
  ).iloc[0]

  # Dicionário com tipos nativos do Python (float, int, str)
  estatisticas = {
      "resumo_geral": {
          "receita_total_geral": round(
              float(df_transformado["receita_total"].sum()), 2
          ),
          "total_transacoes": int(len(df_transformado)),
          "quantidade_total_itens": int(df_transformado["quantidade"].sum()),
          "ticket_medio_geral": round(
              float(df_transformado["receita_total"].mean()), 2
          ),
          "total_clientes_unicos": int(df_transformado["cliente"].nunique()),
      },
      "analise_temporal": {
          "mes_maior_receita": int(mes_top["mes"]),
          "receita_mes_maior": round(float(mes_top["receita_total"]), 2),
      },
      "produtos_e_categorias": {
          "produto_top_1": str(prod_top["produto"]),
          "receita_produto_top_1": round(float(prod_top["receita_total"]), 2),
          "categoria_top_1": str(cat_top["categoria"]),
          "receita_categoria_top_1": round(float(cat_top["receita_total"]), 2),
      },
      "desempenho_regional": {
          "regiao_maior_receita": str(reg_top["regiao"]),
          "receita_regiao_maior": round(float(reg_top["receita_total"]), 2),
          "regiao_maior_ticket_medio": str(reg_ticket_top["regiao"]),
          "ticket_medio_regiao_maior": round(
              float(reg_ticket_top["ticket_medio"]), 2
          ),
      },
      "segmentacao_clientes": {
          "total_clientes_ouro": int((df_clientes["segmento"] == "Ouro").sum()),
          "total_clientes_prata": int(
              (df_clientes["segmento"] == "Prata").sum()
          ),
          "total_clientes_bronze": int(
              (df_clientes["segmento"] == "Bronze").sum()
          ),
          "cliente_maior_gasto": str(
              df_clientes.sort_values("total_gasto", ascending=False).iloc[0][
                  "cliente"
              ]
          ),
      },
  }

  # 3. Exportar para JSON
  caminho_json = "estatisticas_gerais.json"
  with open(caminho_json, "w", encoding="utf-8") as file:
    json.dump(estatisticas, file, indent=4, ensure_ascii=False)

  # 4. Ler de volta o arquivo JSON gravado (Comprovação da gravação)
  with open(caminho_json, "r", encoding="utf-8") as file:
    conferencia_json = json.load(file)

  # 5. Exibição no console
  print("\n")
  print("EXPORTAÇÃO DE DADOS E LEITURA JSON:")
  print("\n")
  print("Arquivos CSV salvos com sucesso:")
  print("  - metricas_por_mes.csv")
  print("  - segmentacao_clientes.csv")
  print("\n Conteúdo lido de estatisticas_gerais.json:")
  print(json.dumps(conferencia_json, indent=4, ensure_ascii=False))


# Executar a exportação
exportar_resultados(
    dicionario_metricas, df_clientes_segmentados, df_transformado
)



EXPORTAÇÃO DE DADOS E LEITURA JSON:


Arquivos CSV salvos com sucesso:
  - metricas_por_mes.csv
  - segmentacao_clientes.csv

 Conteúdo lido de estatisticas_gerais.json:
{
    "resumo_geral": {
        "receita_total_geral": 1380210.31,
        "total_transacoes": 194,
        "quantidade_total_itens": 1035,
        "ticket_medio_geral": 7114.49,
        "total_clientes_unicos": 49
    },
    "analise_temporal": {
        "mes_maior_receita": 2,
        "receita_mes_maior": 186065.22
    },
    "produtos_e_categorias": {
        "produto_top_1": "Notebook",
        "receita_produto_top_1": 493242.63,
        "categoria_top_1": "Computadores",
        "receita_categoria_top_1": 698455.24
    },
    "desempenho_regional": {
        "regiao_maior_receita": "Centro-Oeste",
        "receita_regiao_maior": 342981.92,
        "regiao_maior_ticket_medio": "Nordeste",
        "ticket_medio_regiao_maior": 8653.08
    },
    "segmentacao_clientes": {
        "total_clientes_ouro": 35,
        "

In [ ]:
#RF10 – Executar o Fluxo Completo (Ponto de Entrada)

import os


def main():
  """Executa o fluxo completo do SalesInsight PY do início ao fim."""
  print("\n")
  print("      SALESINSIGHT PY - Análise de Dados de Vendas")
  print("\n")

  # ETAPA 0: Garantir a existência do dataset bruto (RF01)

  if not os.path.exists("vendas.csv"):
    print("\n[ETAPA 0] Criando dataset sintético 'vendas.csv'...")
    df_bruto = gerar_dataset_vendas(n_registros=200, seed=42)
    df_bruto.to_csv("vendas.csv", index=False)
  else:
    print("\n[ETAPA 0] Carregando dataset existente 'vendas.csv'...")
    df_bruto = pd.read_csv("vendas.csv")


  # ETAPA 1: Inspeção Inicial dos Dados (RF02)

  print("\n[ETAPA 1] Inspecionando estrutura dos dados...")
  inspecionar_dados(df_bruto)

  # ETAPA 2: Limpeza e Tratamento com Datetime (RF03)

  print("\n[ETAPA 2] Realizando limpeza e conversão de datas...")
  df_limpo, relatorio_limpeza = tratar_dados(df_bruto)

  # ETAPA 3: Criar Colunas Derivadas (RF04)

  print("\n[ETAPA 3] Criando colunas derivadas e temporais...")
  df_transformado = criar_colunas_derivadas(df_limpo)

  # ETAPA 4: Agregações com Groupby (RF05)

  print("\n[ETAPA 4] Calculando métricas agregadas...")
  dicionario_metricas = calcular_metricas(df_transformado)

  # ETAPA 5: Segmentação de Clientes com Lambda (RF06)

  print("\n[ETAPA 5] Segmentando clientes em Bronze, Prata e Ouro...")
  df_clientes_segmentados = segmentar_clientes(df_transformado)

  # ETAPA 6: Demonstração de Função de Ordem Superior (RF08)

  print("\n[ETAPA 6] Aplicando Função de Ordem Superior...")
  df_transformado = processar_coluna(
      df_transformado,
      coluna="receita_total",
      funcao_transformacao=lambda x: round(x / 1000, 2),
      nome_saida="receita_em_milhares",
  )

  # ETAPA 7: Gerar e Salvar Visualizações (RF07)

  print("\n[ETAPA 7] Gerando e salvando gráficos em PNG...")
  gerar_graficos(df_transformado, dicionario_metricas)

  # ETAPA 8: Exportar Resultados em CSV/JSON e Reler JSON (RF09)

  print("\n[ETAPA 8] Exportando CSVs e JSON com releitura de checagem...")
  exportar_resultados(
      dicionario_metricas, df_clientes_segmentados, df_transformado
  )

  print("\n")
  print("   [CONCLUÍDO] Fluxo do SalesInsight PY executado com sucesso!")
  print("\n")


# Bloco de execução principal
if __name__ == "__main__":
  main()



      SALESINSIGHT PY - Análise de Dados de Vendas



[ETAPA 0] Carregando dataset existente 'vendas.csv'...

[ETAPA 1] Inspecionando estrutura dos dados...

INSPECAO INICIAL DO DATASET:

Shape: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade          int64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        0
preco_unitario    0
dtype: int64

Primeiros registros:
   id_venda     data_venda      cliente     produto     categoria    regiao  \
0         1     2025-05-21  Cliente_016       Mouse   Perifericos   Sudeste   
1         2  DATA INVALIDA  Cliente_003       Mouse   Perifer